# Lecture 11: Node and Graph Classification — Exercises

Companion to the note [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).

Practise node and graph classification: attention variants and multi-head attention, label propagation (Zhu & Ghahramani) in closed form, the Iterative Classification Algorithm, the self-study's DeepWalk-PPMI pipeline with repeated stratified CV, a 2-layer GCN trained from scratch with hand-written backprop, graph-level read-outs (sum/mean pooling) and coarsening with $S^\top AS$. Graphs are NumPy adjacency matrices; no torch or networkx.

**17 exercises** · 🧠 Concept ×6 · ✍️ By hand ×5 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

# Zachary's Karate Club (same 0-indexed edge list as networkx.karate_club_graph(), 34 nodes, 78 edges)
KARATE_EDGES = [(0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 10), (0, 11), (0, 12), (0, 13),
    (0, 17), (0, 19), (0, 21), (0, 31), (1, 2), (1, 3), (1, 7), (1, 13), (1, 17), (1, 19), (1, 21), (1, 30), (2, 3),
    (2, 7), (2, 8), (2, 9), (2, 13), (2, 27), (2, 28), (2, 32), (3, 7), (3, 12), (3, 13), (4, 6), (4, 10), (5, 6),
    (5, 10), (5, 16), (6, 16), (8, 30), (8, 32), (8, 33), (9, 33), (13, 33), (14, 32), (14, 33), (15, 32), (15, 33),
    (18, 32), (18, 33), (19, 33), (20, 32), (20, 33), (22, 32), (22, 33), (23, 25), (23, 27), (23, 29), (23, 32),
    (23, 33), (24, 25), (24, 27), (24, 31), (25, 31), (26, 29), (26, 33), (27, 33), (28, 31), (28, 33), (29, 32),
    (29, 33), (30, 32), (30, 33), (31, 32), (31, 33), (32, 33)]
# Ground-truth split after the club broke up: 0 = "Mr. Hi", 1 = "Officer"
KARATE_CLUB = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

def edges_to_adj(edges, n=None, directed=False):
    """Edge list -> dense 0/1 NumPy adjacency matrix."""
    n = n if n is not None else max(max(e) for e in edges) + 1
    A = np.zeros((n, n))
    for u, v in edges:
        A[u, v] = 1
        if not directed:
            A[v, u] = 1
    return A

A_karate = edges_to_adj(KARATE_EDGES, 34)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold
rng = np.random.default_rng(11)
relu = lambda x: np.maximum(x, 0)

def softmax(Z, axis=-1):
    Z = Z - Z.max(axis, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis, keepdims=True)

# Attributed SBM for node classification: 4 classes x 30 nodes, noisy 8-d features
_MU = np.random.default_rng(0).choice([-1.0, 1.0], size=(4, 8))
def make_sbm(seed, n=120, p_in=0.2, p_out=0.02, noise=1.5):
    r = np.random.default_rng(seed)
    blk = np.repeat(np.arange(4), n // 4)
    P = np.where(blk[:, None] == blk[None, :], p_in, p_out)
    A = np.triu((r.random((n, n)) < P).astype(float), 1); A = A + A.T
    X = _MU[blk] + noise * r.normal(size=(n, 8))
    return A, X, blk

## Part A · Concepts

### Exercise 1 · Node vs graph classification, transductive vs inductive
*🧠 Concept · ★☆☆*

1. Give one example each of node classification and graph classification. What extra component does graph classification need?
2. Classify these as transductive or inductive and justify: label propagation, DeepWalk-PPMI + logistic regression (self-study Ex. 1),
   GraphSAGE, a 2-layer GCN with fixed weights applied to a new citation graph.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ask: if a brand-new node (or graph) arrives, can the method produce a prediction without re-running the whole optimisation/inversion?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Node: predict a paper's topic in CORA, or which faction a karate member joins. Graph: predict whether a molecule is toxic. Graph
   classification needs a **read-out / pooling** step that turns a variable number of node embeddings into one fixed-size vector $\mathbf z_{\mathcal G}$.
2. * Label propagation: **transductive**, since $(I-P_{uu})^{-1}$ is computed for this specific graph and must be recomputed when nodes are added.
   * DeepWalk-PPMI + LR: **transductive**, because the embeddings are computed from the whole graph $A$ (test nodes included, but not their labels).
   * GraphSAGE: **inductive**, with learnt aggregation weights applied to any neighbourhood.
   * GCN with fixed weights on a new graph: **inductive**, as the weights act on features and neighbourhoods, not on ids (cf. Lecture 10, Ex. 17).

</details>

### Exercise 2 · Node classification is not link prediction
*🧠 Concept · ★☆☆*

Critical Insight 1 warns that the best link predictor is not automatically the best node classifier. Compare the two tasks on:
(a) what is masked in the train/test split, (b) the decoder, (c) the need for negative sampling, (d) the geometry the embeddings should have.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Node classification hides labels; link prediction hides edges.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| | node classification | link prediction |
|---|---|---|
| split | mask **labels**; topology stays intact | remove **edges** from $A$; the graph itself changes |
| decoder | class head + softmax on $\mathbf z_u$ | pairwise score $\sigma(\mathbf z_u^\top\mathbf z_v)$ or MLP |
| negatives | not needed (labels are explicit) | essential: sample non-edges as 0 targets |
| geometry | **homophily**: same-class nodes in compact, separable clusters | **proximity**: linked pairs close; positional/role info matters |

A Node2Vec embedding tuned to reconstruct edges may mix classes inside a community boundary, while a classifier wants linearly
separable class clusters, so model selection must be done per task.

</details>

### Exercise 3 · Attention variants and multi-head attention
*🧠 Concept · ★★☆*

1. Compare the three attention score functions in the note (GAT concatenation, bilinear $\mathbf h_u^\top W\mathbf h_v$, MLP) in terms of
   parameters and expressiveness.
2. Why use $K$ heads, and why does the self-study's GAT **concatenate** heads in the hidden layer but **average** them in the output layer?
3. How is GNN attention related to a Transformer's self-attention?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A sequence Transformer is attention over a *complete* graph of tokens.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. GAT: $\mathbf a\in\mathbb R^{2d'}$ plus $W$, an additive score $\mathbf a_1^\top W\mathbf h_u + \mathbf a_2^\top W\mathbf h_v$ inside a non-linearity: cheap but
   "static" ranking of neighbours. Bilinear: $d^2$ parameters, a multiplicative interaction (like Transformer query–key products). MLP: the most
   flexible and the most expensive.
2. Several heads learn different notions of relevance and average out the variance of a single softmax, which stabilises training. Hidden layers concatenate
   ($K\cdot d'$ features: richer representation, e.g. $8\times8 = 64$); the output layer must produce exactly $C$ class logits, so heads are averaged
   (`concat=False`).
3. A Transformer's self-attention is GAT-style attention where every token attends to every other token, i.e. message passing on the complete graph
   (with positional encodings supplying the structure). Graph attention restricts the softmax to $\mathcal N(u)$, injecting the graph as an inductive bias.

</details>

### Exercise 4 · How ICA works, and how it fails
*🧠 Concept · ★★☆*

Explain the Iterative Classification Algorithm step by step: what goes into the feature vector $\Phi_u$, why the classifier has to be applied
repeatedly, and why the first step needs a separate bootstrap. Name one failure mode.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Relational features of $u$ depend on the (unknown) labels of $u$'s neighbours.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\Phi_u$ = intrinsic features $x_u$ ⊕ **relational** features summarising neighbour labels (e.g. the fraction of neighbours in each class).
1. *Bootstrap*: unlabeled neighbours have no labels yet, so assign initial labels with an intrinsic-only classifier (or use only labeled neighbours).
2. Train the base classifier on the labeled nodes using $\Phi$.
3. Repeat: recompute $\Phi_u$ from current labels (true for labeled, predicted for unlabeled), re-predict the unlabeled nodes, until labels stop changing.

The iteration is needed because predictions are mutually dependent: changing one node's label changes its neighbours' features.
Failure modes: **error propagation/cascades** (a wrong bootstrap label pulls its neighbours along), oscillation (no convergence guarantee), and
it's useless on **heterophilous** graphs where neighbours tend to have different labels (unless the classifier learns that inverse relationship).

</details>

### Exercise 5 · Pooling: flat, attention, coarsening
*🧠 Concept · ★★☆*

1. Give two graphs that **mean** pooling cannot distinguish but **sum** pooling can, and vice versa explain when mean is preferable.
2. What does the LSTM/attention read-out ($q_t$, $a_{v,t}$, $o_t$) add over flat pooling?
3. In coarsening $A^{new} = S^\top AS$, what do the diagonal and off-diagonal entries of $A^{new}$ mean when $S$ is a hard 0/1 assignment?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Take two graphs whose nodes all have identical embeddings but different sizes.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. A triangle and a 6-cycle with constant node features $\mathbf z_u = \mathbf c$ (2-regular, so even after message passing all embeddings are equal):
   mean gives $\mathbf c$ for both, sum gives $3\mathbf c$ vs $6\mathbf c$. Mean is preferable when the size should *not* matter (e.g. the
   fraction of a property), and is numerically more stable across very different graph sizes.
2. The read-out becomes **content-dependent**: at each step $t$ an LSTM query $q_t$ decides which nodes to attend to; $T$ steps can focus on different
   parts of the graph, and $o_1\oplus\dots\oplus o_T$ keeps them separate (Set2Set). It's still permutation invariant, since the softmax runs over a set.
3. $A^{new}_{ij} = \sum_{u\in C_i}\sum_{v\in C_j} A_{uv}$: off-diagonal = number of edges between clusters $i$ and $j$; diagonal = twice the
   number of edges inside cluster $i$ (each undirected edge counted in both directions). Coarsening applied repeatedly yields a hierarchy (DiffPool
   learns a soft $S$ with a GNN).

</details>

### Exercise 6 · Reading the self-study results
*🧠 Concept · ★★☆*

1. Why did the self-study evaluate the karate classifier with `RepeatedStratifiedKFold(n_splits=5, n_repeats=20)` rather than a single split?
2. Why did the 3-layer GCN do *worse* than the 2-layer one on Cora?
3. What does swapping ReLU for ELU change?
4. Is the DeepWalk-PPMI pipeline leaking test labels? Test *structure*?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

34 nodes, 5 folds → about 7 test nodes per fold.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. With ~7 test nodes per fold, one misclassification moves accuracy by ~14 points. Repeating 20 times with stratification (both classes in every fold)
   averages 100 estimates and gives a stable mean plus an honest spread.
2. **Over-smoothing**: each GCN layer is a Laplacian smoothing filter; 3 hops on Cora's small-world graph already mixes many classes, making node
   representations less separable, and the extra layer adds parameters that overfit the 140 training labels.
3. ELU $= \alpha(e^x-1)$ for $x<0$ keeps a non-zero gradient for negative inputs (no "dead" units) and pushes mean activations towards 0.
4. Labels: no, the logistic regression is trained only on the training folds. Structure: yes, by design; the embeddings see the whole graph
   including test nodes' edges. That's acceptable in the **transductive** setting (the graph is given, labels are what's missing).

</details>

## Part B · By hand

### Exercise 7 · Label propagation in closed form
*✍️ By hand · ★★☆*

Path graph $0-1-2-3$. Node 0 is labeled class A, node 3 class B; nodes 1, 2 are unlabeled. Order the nodes as $V_l = (0, 3)$, $V_u = (1, 2)$.

1. Write $P_{uu}$ and $P_{ul}$ from the random-walk matrix $P = D^{-1}A$.
2. Compute $Y_u = (I - P_{uu})^{-1}P_{ul}Y_l$ with $Y_l = \begin{pmatrix}1&0\\0&1\end{pmatrix}$ (one-hot rows for A, B).
3. Interpret the entries as absorption probabilities.

In [ ]:
Y_u_hand = None   # 2x2 array: rows = nodes 1, 2; columns = P(A), P(B)

# reference: clamp labeled nodes and iterate the random walk until convergence
_A = edges_to_adj([(0, 1), (1, 2), (2, 3)], 4); _P = _A / _A.sum(1, keepdims=True)
_Y = np.zeros((4, 2)); _Y[0, 0] = 1; _Y[3, 1] = 1
for _ in range(500):
    _Y = _P @ _Y; _Y[0] = [1, 0]; _Y[3] = [0, 1]
check('Y_u', Y_u_hand, _Y[[1, 2]], tol=1e-6)

<details>
<summary><b>💡 Hint 1</b></summary>

$P_{uu} = \begin{pmatrix}0 & \tfrac12\\ \tfrac12 & 0\end{pmatrix}$, $P_{ul} = \begin{pmatrix}\tfrac12 & 0\\ 0 & \tfrac12\end{pmatrix}$ (columns: node 0, node 3).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$(I - P_{uu})^{-1} = \frac{1}{3/4}\begin{pmatrix}1 & \tfrac12\\ \tfrac12 & 1\end{pmatrix}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(I-P_{uu})^{-1} = \begin{pmatrix}4/3 & 2/3\\ 2/3 & 4/3\end{pmatrix}$, $P_{ul}Y_l = \begin{pmatrix}1/2 & 0\\ 0 & 1/2\end{pmatrix}$ →
$Y_u = \begin{pmatrix}2/3 & 1/3\\ 1/3 & 2/3\end{pmatrix}$.
A random walk starting at node 1 is absorbed at node 0 (label A) with probability $2/3$ and at node 3 with $1/3$, the harmonic
solution (each unlabeled value is the average of its neighbours). Node 1 → A, node 2 → B.

```python
Y_u_hand = np.array([[2/3, 1/3], [1/3, 2/3]])
```

</details>

### Exercise 8 · Dot-product attention by hand
*✍️ By hand · ★☆☆*

Bilinear attention with $W = I$: $\mathbf h_u = (1, 0)$, neighbours $\mathbf h_{v_1} = (1, 1)$, $\mathbf h_{v_2} = (0, 1)$, $\mathbf h_{v_3} = (-1, 0)$.
Compute the attention weights $\alpha_{u,v}$ and the message $m_{\mathcal N(u)} = \sum_v\alpha_{u,v}\mathbf h_v$.

In [ ]:
alpha_dot = None
m_dot = None

_H = np.array([[1, 1], [0, 1], [-1, 0.]]); _s = _H @ np.array([1, 0.])
_al = np.exp(_s) / np.exp(_s).sum()
check('attention weights', alpha_dot, _al, tol=1e-4); check('message', m_dot, _al @ _H, tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Scores $\mathbf h_u^\top\mathbf h_v = 1, 0, -1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Softmax of $(1, 0, -1)$: $e^1 = 2.718$, $e^0 = 1$, $e^{-1} = 0.368$, sum $4.086$ → $\alpha = (0.665, 0.245, 0.090)$.
$m = 0.665(1,1) + 0.245(0,1) + 0.090(-1,0) = (0.575, 0.910)$. The neighbour most aligned with $\mathbf h_u$ dominates.

```python
_s = np.array([1., 0., -1.])
alpha_dot = np.exp(_s) / np.exp(_s).sum()
m_dot = alpha_dot @ np.array([[1, 1], [0, 1], [-1, 0.]])
```

</details>

### Exercise 9 · Flat pooling by hand
*✍️ By hand · ★☆☆*

Node embeddings after message passing:
$G_1$: $\{(1, 0), (1, 0), (0, 2)\}$; $G_2$: $\{(1, 0), (0, 2)\}$ (the same "types" of nodes, but $G_1$ has an extra copy).
Compute the sum and mean read-outs of both graphs. Which read-out separates them?

In [ ]:
sum_G1 = None; sum_G2 = None
mean_G1 = None; mean_G2 = None

_G1 = np.array([[1, 0], [1, 0], [0, 2.]]); _G2 = np.array([[1, 0], [0, 2.]])
check('sum G1', sum_G1, _G1.sum(0)); check('sum G2', sum_G2, _G2.sum(0))
check('mean G1', mean_G1, _G1.mean(0)); check('mean G2', mean_G2, _G2.mean(0))

<details>
<summary><b>💡 Hint</b></summary>

$\mathbf z_{\mathcal G} = \sum_u\mathbf z_u / f_n(|\mathcal V|)$ with $f_n = 1$ (sum) or $f_n = |\mathcal V|$ (mean).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Sum: $G_1 = (2, 2)$, $G_2 = (1, 2)$. Mean: $G_1 = (0.667, 0.667)$, $G_2 = (0.5, 1)$. Here **both** separate them, but only sum
keeps the counts; if $G_2$ were $\{(1,0),(1,0),(0,2),(0,2)\}$ vs $\{(1,0),(0,2)\}$ the means would coincide while the sums differ. Sum pooling is
the injective choice (as in GIN), which matters for the expressiveness discussion in Lecture 12.

```python
sum_G1 = [2, 2]; sum_G2 = [1, 2]
mean_G1 = [2/3, 2/3]; mean_G2 = [0.5, 1]
```

</details>

### Exercise 10 · Coarsening a graph by hand
*✍️ By hand · ★☆☆*

Path graph $0-1-2-3$ with node features $X = (1, 2, 3, 4)^\top$. Cluster $\{0,1\}$ and $\{2,3\}$ with the hard assignment
$S = \begin{pmatrix}1&0\\1&0\\0&1\\0&1\end{pmatrix}$. Compute $A^{new} = S^\top AS$ and $X^{new} = S^\top X$.

In [ ]:
A_new_hand = None
X_new_hand = None

_A = edges_to_adj([(0, 1), (1, 2), (2, 3)], 4); _S = np.array([[1, 0], [1, 0], [0, 1], [0, 1.]])
check('A_new', A_new_hand, _S.T @ _A @ _S); check('X_new', X_new_hand, _S.T @ np.array([1, 2, 3, 4.]))

<details>
<summary><b>💡 Hint</b></summary>

$A^{new}_{ij}$ sums the block of $A$ with rows in cluster $i$ and columns in cluster $j$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$A^{new} = \begin{pmatrix}2 & 1\\ 1 & 2\end{pmatrix}$: each cluster has one internal edge (counted twice on the diagonal) and one edge $(1,2)$ links them.
$X^{new} = (1+2,\ 3+4) = (3, 7)$: features are summed per cluster (use $S$ with columns normalised by cluster size to get means).

```python
A_new_hand = np.array([[2, 1], [1, 2]])
X_new_hand = [3, 7]
```

</details>

### Exercise 11 · Multi-head GAT dimensions on Cora
*✍️ By hand · ★☆☆*

The self-study's GAT on Cora: 1433 input features, `hidden_channels = 8`, `heads = 8` (concatenated), and an output layer with 7 classes and
`heads = 1, concat=False`.

1. What is the dimension of a node's representation after the first layer?
2. How many entries do the linear transforms $W_k$ have in each layer (ignore biases and attention vectors)?
3. How many attention-vector parameters $\mathbf a_k\in\mathbb R^{2d'}$ does the first layer have?

In [ ]:
hidden_dim = None
W_params_layer1 = None
W_params_layer2 = None
att_params_layer1 = None

check('hidden dim', hidden_dim, 8 * 8)
check('layer 1 W', W_params_layer1, 1433 * 8 * 8); check('layer 2 W', W_params_layer2, 64 * 7)
check('layer 1 attention', att_params_layer1, 8 * 2 * 8)

<details>
<summary><b>💡 Hint</b></summary>

Each head $k$ has its own $W_k\in\mathbb R^{8\times1433}$ and $\mathbf a_k\in\mathbb R^{16}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $8\text{ heads}\times8 = 64$.
2. Layer 1: $8\cdot(1433\cdot8) = 91\,712$; layer 2: $64\cdot7 = 448$.
3. $8\cdot16 = 128$. Almost all parameters sit in the first projection of the sparse bag-of-words input, which is why dropout (0.5 in
   the self-study) on inputs and attention is so important with only 140 labelled training nodes.

```python
hidden_dim = 64
W_params_layer1 = 1433 * 64
W_params_layer2 = 448
att_params_layer1 = 128
```

</details>

## Part C · Code from scratch

### Exercise 12 · Label propagation on karate
*💻 Code · ★★☆*

Only two members are labeled: the instructor (node 0, class 0) and the officer (node 33, class 1). Implement
`label_propagation(A, labeled, y_l)` returning the class-probability matrix for **all** nodes using the closed form
$Y_u = (I-P_{uu})^{-1}P_{ul}Y_l$ (use `np.linalg.solve`). Store the predicted classes in `lp_pred` and the accuracy on the 32 unlabeled
nodes in `lp_acc`.

In [ ]:
def label_propagation(A, labeled, y_l, n_classes=2):
    # labeled: array of node ids, y_l: their classes
    # TODO: return an (n, n_classes) matrix of class probabilities
    return None

lp_probs = label_propagation(A_karate, np.array([0, 33]), np.array([0, 1]))
lp_pred = None
lp_acc = None

# reference: iterative propagation with clamping
_P = A_karate / A_karate.sum(1, keepdims=True)
_Y = np.zeros((34, 2)); _Y[0, 0] = 1; _Y[33, 1] = 1
for _ in range(3000):
    _Y = _P @ _Y; _Y[0] = [1, 0]; _Y[33] = [0, 1]
check('LP probabilities', lp_probs, _Y, tol=1e-5)
_mask = np.ones(34, bool); _mask[[0, 33]] = False
check('LP accuracy', lp_acc, np.mean(_Y.argmax(1)[_mask] == KARATE_CLUB[_mask]))

<details>
<summary><b>💡 Hint 1</b></summary>

`u = np.setdiff1d(np.arange(n), labeled)`; `P = A / A.sum(1, keepdims=True)`; `P[np.ix_(u, u)]`, `P[np.ix_(u, labeled)]`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`Y_l = np.eye(n_classes)[y_l]`; `Y_u = np.linalg.solve(np.eye(len(u)) - P_uu, P_ul @ Y_l)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With only 2 of 34 labels, label propagation classifies 31 of the 32 others correctly (only node 8, a member with ties to both leaders, is wrong). The method has no parameters
to learn: it relies purely on homophily. The price is transduction: adding a member means solving a new linear system
(Critical Insight brainstorm box).

```python
def label_propagation(A, labeled, y_l, n_classes=2):
    n = len(A)
    u = np.setdiff1d(np.arange(n), labeled)
    P = A / A.sum(1, keepdims=True)
    Y_l = np.eye(n_classes)[y_l]
    Y_u = np.linalg.solve(np.eye(len(u)) - P[np.ix_(u, u)], P[np.ix_(u, labeled)] @ Y_l)
    Y = np.zeros((n, n_classes)); Y[labeled] = Y_l; Y[u] = Y_u
    return Y

lp_probs = label_propagation(A_karate, np.array([0, 33]), np.array([0, 1]))
lp_pred = lp_probs.argmax(1)
_mask = np.ones(34, bool); _mask[[0, 33]] = False
lp_acc = np.mean(lp_pred[_mask] == KARATE_CLUB[_mask])
print('accuracy on unlabeled nodes', lp_acc, ' wrong:', np.flatnonzero(lp_pred != KARATE_CLUB))
```

</details>

### Exercise 13 · Iterative Classification Algorithm
*💻 Code · ★★★*

Karate members get weak noisy features `X_ica` (below). For a split with 5 labeled nodes per class:

1. **Intrinsic baseline**: logistic regression on `X_ica` only.
2. **ICA**: bootstrap with the baseline's predictions, then repeat (≤ 10 times): $\Phi_u = [x_u,\ \text{fraction of } u\text{'s neighbours currently in class 1}]$,
   fit `LogisticRegression` on the labeled nodes, re-predict unlabeled nodes (labeled nodes keep their true labels), stop when nothing changes.

Write `ica(seed)` returning `(acc_baseline, acc_ica)` on the unlabeled nodes, run it for seeds `0..19` and store the mean accuracies in
`acc_base_mean`, `acc_ica_mean`.

In [ ]:
X_ica = (2 * KARATE_CLUB - 1)[:, None] * 0.5 + np.random.default_rng(0).normal(size=(34, 2))

def ica(seed, max_iter=10):
    r = np.random.default_rng(seed)
    lab = np.zeros(34, bool)
    lab[r.choice(np.flatnonzero(KARATE_CLUB == 0), 5, replace=False)] = True
    lab[r.choice(np.flatnonzero(KARATE_CLUB == 1), 5, replace=False)] = True
    # TODO
    return None

acc_base_mean = None
acc_ica_mean = None

if acc_ica_mean is not None:
    check('ICA improves on the intrinsic baseline by > 5 points', acc_ica_mean > acc_base_mean + 0.05, True)
    _r = np.random.default_rng(0); _lab = np.zeros(34, bool)
    _lab[_r.choice(np.flatnonzero(KARATE_CLUB == 0), 5, replace=False)] = True
    _lab[_r.choice(np.flatnonzero(KARATE_CLUB == 1), 5, replace=False)] = True
    _b = LogisticRegression().fit(X_ica[_lab], KARATE_CLUB[_lab]).predict(X_ica)
    check('baseline accuracy (seed 0)', ica(0)[0], np.mean(_b[~_lab] == KARATE_CLUB[~_lab]))
else:
    check('ICA', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

Relational feature: `(A_karate @ cur) / A_karate.sum(1)` where `cur` is the current 0/1 label vector.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`new = np.where(lab, KARATE_CLUB, clf.predict(Phi))`; break if `np.array_equal(new, cur)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Averaged over 20 random splits the intrinsic classifier reaches about 71% and ICA about 82%: neighbours' (predicted) labels add the
homophily signal that the noisy features lack. Individual splits vary a lot; sometimes ICA doesn't help at all, because a bad bootstrap
can lock in a wrong neighbourhood majority (error cascade). Averaging over many splits is essential on a 34-node graph.

```python
X_ica = (2 * KARATE_CLUB - 1)[:, None] * 0.5 + np.random.default_rng(0).normal(size=(34, 2))

def ica(seed, max_iter=10):
    r = np.random.default_rng(seed)
    lab = np.zeros(34, bool)
    lab[r.choice(np.flatnonzero(KARATE_CLUB == 0), 5, replace=False)] = True
    lab[r.choice(np.flatnonzero(KARATE_CLUB == 1), 5, replace=False)] = True
    y = KARATE_CLUB
    base = LogisticRegression().fit(X_ica[lab], y[lab])
    cur = np.where(lab, y, base.predict(X_ica))
    acc_base = np.mean(cur[~lab] == y[~lab])
    deg = A_karate.sum(1)
    for _ in range(max_iter):
        Phi = np.c_[X_ica, (A_karate @ cur) / deg]
        clf = LogisticRegression().fit(Phi[lab], y[lab])
        new = np.where(lab, y, clf.predict(Phi))
        if np.array_equal(new, cur):
            break
        cur = new
    return acc_base, np.mean(cur[~lab] == y[~lab])

res = np.array([ica(s) for s in range(20)])
acc_base_mean, acc_ica_mean = res.mean(0)
print('intrinsic only', acc_base_mean, ' ICA', acc_ica_mean)
```

</details>

### Exercise 14 · DeepWalk-PPMI embeddings (self-study Exercise 1)
*💻 Code · ★★☆*

Re-implement the self-study pipeline:

1. 20 uniform random walks of length 40 from every node; count co-occurrences $C_{ij}$ within a window of 5 (symmetric).
2. $\text{PPMI}(i,j) = \max\left(\log\frac{C_{ij}\sum_{ab}C_{ab}}{(\sum_bC_{ib})(\sum_aC_{aj})}, 0\right)$ (with the note's `eps` trick).
3. $\mathbf Z = U_{16}\Sigma_{16}^{1/2}$ from the SVD of the PPMI matrix.
4. Evaluate `make_pipeline(StandardScaler(), LogisticRegression())` with `RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=0)`.
Store the mean accuracy in `ppmi_acc`.

In [ ]:
def ppmi(C, eps=1e-12):
    # TODO
    return None

C_walk = None
Z_ppmi = None
ppmi_acc = None

_C = np.array([[0, 2, 1], [2, 0, 1], [1, 1, 0.]])
_ref = np.maximum(np.log(_C * _C.sum() / np.outer(_C.sum(1), _C.sum(0)) + 1e-300), 0)
_got = ppmi(_C)
check('PPMI on a small matrix', None if _got is None else np.where(_C > 0, _got, 0), np.where(_C > 0, _ref, 0), tol=1e-6)
check('DeepWalk-PPMI accuracy >= 0.85', None if ppmi_acc is None else ppmi_acc >= 0.85, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Co-occurrence: for every walk `w` and position `i`, for `j` in the window, `C[w[i], w[j]] += 1` (skip `j == i`).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`np.maximum(np.log((C * total + eps) / (row @ col + eps)), 0)` with `row = C.sum(1, keepdims=True)`, `col = C.sum(0, keepdims=True)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Accuracy is around 0.87 (it varies a little with the walk seed). This is the explicit matrix-factorisation view of DeepWalk (Lecture 9, NetMF): PPMI rewards node pairs
that co-occur more often than their marginal frequencies predict, which down-weights hubs that co-occur with everyone. Clipping at 0
(the "positive" in PPMI) discards uninformative negative associations and keeps the matrix sparse-ish.

```python
def ppmi(C, eps=1e-12):
    total = C.sum()
    row = C.sum(1, keepdims=True); col = C.sum(0, keepdims=True)
    return np.maximum(np.log((C * total + eps) / (row @ col + eps)), 0.0)

r = np.random.default_rng(0)
nbrs = [np.flatnonzero(A_karate[i]) for i in range(34)]
C_walk = np.zeros((34, 34))
for _ in range(20):
    for s in range(34):
        w = [s]
        for _ in range(39):
            w.append(r.choice(nbrs[w[-1]]))
        for i in range(len(w)):
            for j in range(max(0, i - 5), min(len(w), i + 6)):
                if j != i:
                    C_walk[w[i], w[j]] += 1
U, s, _ = np.linalg.svd(ppmi(C_walk), full_matrices=False)
Z_ppmi = U[:, :16] * np.sqrt(s[:16])
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=0)
ppmi_acc = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), Z_ppmi, KARATE_CLUB, cv=cv).mean()
print('DeepWalk-PPMI accuracy', ppmi_acc)
```

</details>

### Exercise 15 · A 2-layer GCN with hand-written backprop
*💻 Code · ★★★*

On an attributed SBM (`A_nc, X_nc, y_nc = make_sbm(seed=5)`, 4 classes) train the classic GCN
$$\hat Y = \text{softmax}\big(\hat A\,\text{ReLU}(\hat AXW_1)\,W_2\big),\qquad \hat A = \tilde D^{-1/2}(A+I)\tilde D^{-1/2},$$
with cross-entropy on the **training mask** only (5 nodes per class). Use full-batch gradient descent (lr 0.2, 300 epochs, hidden 16,
$W\sim\mathcal N(0, 0.1^2)$). Derive and implement the backward pass:
$G_2 = (\hat Y - Y)\odot\text{mask}/n_{train}$, $\nabla W_2 = (\hat AH_1)^\top G_2$, $G_1 = (\hat A^\top G_2W_2^\top)\odot\mathbb 1[\hat AXW_1 > 0]$,
$\nabla W_1 = (\hat AX)^\top G_1$.

Write `train_gcn(A_hat, X, y, train_mask, seed=0)` returning predictions for all nodes. Compare with the same network using $\hat A = I$
(an MLP that ignores the graph). Store the test accuracies in `acc_gcn`, `acc_mlp`.

In [ ]:
A_nc, X_nc, y_nc = make_sbm(seed=5)
train_mask = np.zeros(120, bool)
for c in range(4):
    train_mask[np.flatnonzero(y_nc == c)[:5]] = True

def sym_norm(A):
    At = A + np.eye(len(A)); d = At.sum(1)
    return At / np.sqrt(np.outer(d, d))

def train_gcn(A_hat, X, y, train_mask, seed=0, hidden=16, lr=0.2, epochs=300):
    # TODO: return predicted class for every node
    return None

acc_gcn = None
acc_mlp = None

if acc_gcn is not None:
    check('GCN test accuracy > 0.8', acc_gcn > 0.8, True)
    check('GCN beats the graph-free MLP by > 15 points', acc_gcn > acc_mlp + 0.15, True)
else:
    check('GCN', None, 0)

<details>
<summary><b>💡 Hint 1</b></summary>

Forward: `AX = A_hat @ X; P1 = AX @ W1; H1 = relu(P1); AH = A_hat @ H1; Yhat = softmax(AH @ W2)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`Y = np.eye(4)[y]`; `G2 = (Yhat - Y) * train_mask[:, None] / train_mask.sum()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With only 20 labels the MLP must classify from noisy features alone (≈ 55% test accuracy), while the GCN averages each node's features
with its neighbours', which mostly share its class (homophily), and reaches ≈ 97%. Graph convolution acts as a denoising filter. The
backward pass is ordinary backprop through two linear maps; $\hat A$ is a fixed matrix, so it just appears transposed in the chain rule.

```python
A_nc, X_nc, y_nc = make_sbm(seed=5)
train_mask = np.zeros(120, bool)
for c in range(4):
    train_mask[np.flatnonzero(y_nc == c)[:5]] = True

def sym_norm(A):
    At = A + np.eye(len(A)); d = At.sum(1)
    return At / np.sqrt(np.outer(d, d))

def train_gcn(A_hat, X, y, train_mask, seed=0, hidden=16, lr=0.2, epochs=300):
    r = np.random.default_rng(seed)
    C = y.max() + 1
    W1 = 0.1 * r.normal(size=(X.shape[1], hidden)); W2 = 0.1 * r.normal(size=(hidden, C))
    Y = np.eye(C)[y]
    AX = A_hat @ X
    for _ in range(epochs):
        P1 = AX @ W1; H1 = relu(P1); AH = A_hat @ H1
        Yhat = softmax(AH @ W2)
        G2 = (Yhat - Y) * train_mask[:, None] / train_mask.sum()
        gW2 = AH.T @ G2
        G1 = (A_hat.T @ G2 @ W2.T) * (P1 > 0)
        gW1 = AX.T @ G1
        W1 -= lr * gW1; W2 -= lr * gW2
    return softmax(A_hat @ relu(AX @ W1) @ W2).argmax(1)

test = ~train_mask
acc_gcn = np.mean(train_gcn(sym_norm(A_nc), X_nc, y_nc, train_mask)[test] == y_nc[test])
acc_mlp = np.mean(train_gcn(np.eye(120), X_nc, y_nc, train_mask)[test] == y_nc[test])
print('GCN', acc_gcn, ' MLP', acc_mlp)
```

</details>

### Exercise 16 · Graph classification with message passing and mean pooling
*💻 Code · ★★★*

Build a dataset of 120 graphs with 30–59 nodes: even indices are Erdős–Rényi graphs (`er_graph`, mean degree ≈ 4), odd indices are
Barabási–Albert graphs (`ba_graph`, $m = 2$, also mean degree ≈ 4). Labels: 0 = ER, 1 = BA.

Graph features ("a GNN without learnt weights"): start from $H^{(0)} = $ degree column, apply $K$ mean-aggregation layers
$H^{(k)} = \tilde D^{-1}\tilde AH^{(k-1)}$, concatenate $[H^{(0)}\parallel\dots\parallel H^{(K)}]$ and **mean-pool** over nodes. Evaluate a
5-fold CV `LogisticRegression` for $K = 0, 1, 2$ and store the accuracies in `gc_acc` (dict). Why does $K\ge1$ help so much?

In [ ]:
def er_graph(n, p, r):
    U = np.triu((r.random((n, n)) < p).astype(float), 1); return U + U.T

def ba_graph(n, m, r):
    A = np.zeros((n, n))
    A[:m + 1, :m + 1] = 1 - np.eye(m + 1)                  # start from a small clique
    for v in range(m + 1, n):                              # preferential attachment
        d = A[:v, :v].sum(1)
        t = r.choice(v, m, replace=False, p=d / d.sum())
        A[v, t] = A[t, v] = 1
    return A

r = np.random.default_rng(0)
graphs, y_graphs = [], []
for i in range(120):
    n = int(r.integers(30, 60))
    graphs.append(er_graph(n, 4 / (n - 1), r) if i % 2 == 0 else ba_graph(n, 2, r))
    y_graphs.append(i % 2)
y_graphs = np.array(y_graphs)

def graph_readout(A, K):
    # TODO: return the mean-pooled [H0 || ... || HK] vector
    return None

gc_acc = None

if gc_acc is not None:
    check('K=1 beats K=0', gc_acc[1] > gc_acc[0], True)
    check('K=1 accuracy > 0.95', gc_acc[1] > 0.95, True)
    _A = graphs[1]; _At = _A + np.eye(len(_A)); _d = _A.sum(1)
    check('readout K=1', graph_readout(_A, 1), [_d.mean(), (_At @ _d / _At.sum(1)).mean()])
else:
    check('graph classification', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

Mean degree is (almost) the same for both classes, so $K = 0$ with mean pooling is nearly blind. What is the *average neighbour degree*?

</details>

<details>
<summary><b>✅ Solution</b></summary>

With $K=0$ mean pooling only sees the average degree, which is about 4 in both classes (≈ 0.8 accuracy thanks to the small seed clique of BA).
One round of mean aggregation gives each node the average degree of its neighbourhood. In a heavy-tailed BA graph this is much larger than the
average degree (**friendship paradox**: your friends have more friends than you, because hubs appear in many neighbourhoods), while in ER graphs
the two are similar. Accuracy jumps to ≈ 0.99. Message passing + flat read-out turns a *distributional* property into a graph-level feature.

```python
def er_graph(n, p, r):
    U = np.triu((r.random((n, n)) < p).astype(float), 1); return U + U.T

def ba_graph(n, m, r):
    A = np.zeros((n, n))
    A[:m + 1, :m + 1] = 1 - np.eye(m + 1)
    for v in range(m + 1, n):
        d = A[:v, :v].sum(1)
        t = r.choice(v, m, replace=False, p=d / d.sum())
        A[v, t] = A[t, v] = 1
    return A

r = np.random.default_rng(0)
graphs, y_graphs = [], []
for i in range(120):
    n = int(r.integers(30, 60))
    graphs.append(er_graph(n, 4 / (n - 1), r) if i % 2 == 0 else ba_graph(n, 2, r))
    y_graphs.append(i % 2)
y_graphs = np.array(y_graphs)

def graph_readout(A, K):
    At = A + np.eye(len(A)); P = At / At.sum(1, keepdims=True)
    H = A.sum(1, keepdims=True); feats = [H]
    for _ in range(K):
        H = P @ H; feats.append(H)
    return np.hstack(feats).mean(0)

gc_acc = {}
for K in (0, 1, 2):
    F = np.array([graph_readout(A, K) for A in graphs])
    gc_acc[K] = cross_val_score(LogisticRegression(max_iter=1000), F, y_graphs, cv=5).mean()
print(gc_acc)
```

</details>

### Exercise 17 · Coarsening karate with a spectral assignment
*💻 Code · ★★☆*

Build a hard assignment $S\in\{0,1\}^{34\times2}$ from the sign of the Fiedler vector of $L = D - A$ (Lecture 9). Compute
$A^{new} = S^\top AS$ and $X^{new} = S^\top X$ with $X$ = the degree column. Store them in `A_coarse`, `X_coarse`, and the number of edges
**cut** by the clustering in `n_cut`. Check: why must the entries of $A^{new}$ sum to $\sum_{ij}A_{ij} = 2m$?

In [ ]:
S_assign = None
A_coarse = None
X_coarse = None
n_cut = None

if A_coarse is not None:
    check('A_coarse sums to 2m', A_coarse.sum(), 2 * len(KARATE_EDGES))
    _f = np.linalg.eigh(np.diag(A_karate.sum(1)) - A_karate)[1][:, 1] > 0
    check('cut edges', n_cut, sum(1 for u, v in KARATE_EDGES if _f[u] != _f[v]))
    check('X_coarse sums to 2m', np.sum(X_coarse), 2 * len(KARATE_EDGES))
else:
    check('coarsening', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

`f = eigvecs[:, 1] > 0`; `S = np.c_[f, ~f].astype(float)`. The cut is the off-diagonal entry $A^{new}_{01}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$A^{new}$ is a $2\times2$ "super-graph": the diagonal holds twice the internal edges of each faction, the off-diagonal the 10 edges between
them (both $A^{new}_{01}$ and $A^{new}_{10}$). Every edge endpoint lands in exactly one block because each row of $S$ has a single 1, so the total is
preserved: $\mathbf 1^\top S^\top AS\mathbf 1 = \mathbf 1^\top A\mathbf 1$. With degree features, $X^{new}$ is the volume of each cluster. DiffPool replaces this
fixed $S$ with a soft, learnt $S = \text{softmax}(\text{GNN}(A, X))$.

```python
L = np.diag(A_karate.sum(1)) - A_karate
f = np.linalg.eigh(L)[1][:, 1] > 0
S_assign = np.c_[f, ~f].astype(float)
A_coarse = S_assign.T @ A_karate @ S_assign
X_coarse = S_assign.T @ A_karate.sum(1, keepdims=True)
n_cut = int(A_coarse[0, 1])
print(A_coarse, X_coarse.ravel(), n_cut)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Lecture Notes 1-12](../Lecture%20Notes%201-12.md).*